# SVM (Out-of-Fold)
This notebook trains a Support Vector Machine (SVM) model to generate out-of-fold (OOF) predictions to be incorporated into the stacking ensemble.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.svm import SVC
import optuna
import warnings

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
print("Loading data...")
train = pd.read_csv('../datasets/train_original.csv')
test = pd.read_csv('../datasets/test.csv')

In [ ]:
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [ ]:
# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

In [ ]:
def apply_mappings(df):
    df_out = df.copy()
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    df_out['gender'] = df_out['gender'].fillna('Unknown')
    return df_out

In [ ]:
X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

In [ ]:
# Feature Engineering
def add_features(df):
    df_out = df.copy()
    denom_screen = df_out['daily_screen_time_hours'].replace(0, 0.001)
    denom_notif = df_out['notifications_per_day'].replace(0, 0.001)

    df_out['social_media_ratio'] = df_out['social_media_hours'] / denom_screen
    df_out['gaming_ratio'] = df_out['gaming_hours'] / denom_screen
    df_out['work_study_ratio'] = df_out['work_study_hours'] / denom_screen
    df_out['app_opens_per_hour'] = df_out['app_opens_per_day'] / denom_screen
    df_out['notifications_to_opens_ratio'] = df_out['app_opens_per_day'] / denom_notif
    df_out['sleep_deficit'] = 8.0 - df_out['sleep_hours']
    return df_out

In [ ]:
X_preprocessed = add_features(X_preprocessed)
X_test_preprocessed = add_features(X_test_preprocessed)

In [ ]:
# Columns
categorical_cols = ['gender']
numeric_cols = [col for col in X_preprocessed.columns if col not in categorical_cols]

In [ ]:
# Advanced Preprocessing Pipeline (Crucial for SVMs)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', QuantileTransformer(output_distribution='normal', random_state=42))
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
# Constants
N_FOLDS = 5
N_TRIALS = 10
cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
def eval_pipeline(model, X_train, y_train):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    scores = []
    for train_idx, val_idx in cv_tune.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        pipeline.fit(X_tr, y_tr)
        preds = pipeline.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
    return np.mean(scores)

### Optuna Tuning

In [ ]:
print("--- Starting Optuna Tuning ---")

In [ ]:
def objective_svm(trial):
    # Optuna will search over C and gamma
    C = trial.suggest_float('C', 0.1, 10.0, log=True)
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])
    
    # NOTE FOR KAGGLE: 
    # SVC scales at O(N^3). Even on Kaggle, training this on 500k rows will likely timeout.
    # We set max_iter=2000 to prevent infinite hanging. You can increase this if Kaggle allows it.
    model = SVC(C=C, gamma=gamma, probability=True, cache_size=2000, max_iter=2000, random_state=42)
    
    # We heavily subsample for tuning, otherwise Optuna will take weeks.
    X_sample = X_preprocessed.sample(20000, random_state=42)
    y_sample = y.loc[X_sample.index]
    return eval_pipeline(model, X_sample, y_sample)

In [ ]:
print("Tuning SVC (on subset)...")
study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=N_TRIALS)
svm_best_params = study_svm.best_params

In [ ]:
# Enforce necessary constants for the final model
svm_best_params.update({'probability': True, 'cache_size': 2000, 'max_iter': 5000, 'random_state': 42})

In [ ]:
os.makedirs('../datasets', exist_ok=True)
with open('../datasets/tuned_parameters_svm.json', 'w') as f:
    json.dump({'svm': svm_best_params}, f, indent=4)
print("Saved SVM tuned parameters.")

### OOF Generation

In [ ]:
print("Generating 5-Fold OOF Predictions with SVM...")

In [ ]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

In [ ]:
oof_train_svm = np.zeros(len(X_preprocessed))
test_preds_svm = np.zeros(len(X_test_preprocessed))

In [ ]:
for fold, (train_idx, val_idx) in enumerate(cv.split(X_preprocessed, y)):
    print(f"--- Fold {fold + 1}/{N_FOLDS} ---")
    
    X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
    
    # Pre-fit preprocessor to avoid repetitive expensive imputation
    print("  Fitting Pipeline...")
    X_tr_prep = preprocessor.fit_transform(X_tr)
    X_val_prep = preprocessor.transform(X_val)
    X_test_prep = preprocessor.transform(X_test_preprocessed)
    
    # SVC Model
    # Warning: even with max_iter=5000, this could take hours per fold on Kaggle!
    clf_svm = SVC(**svm_best_params)
    clf_svm.fit(X_tr_prep, y_tr)
    oof_train_svm[val_idx] = clf_svm.predict_proba(X_val_prep)[:, 1]
    test_preds_svm += clf_svm.predict_proba(X_test_prep)[:, 1] / N_FOLDS
    print(f"  SVM Fold {fold+1} AUC: {roc_auc_score(y_val, oof_train_svm[val_idx]):.5f}")

In [ ]:
os.makedirs('../datasets/oof_preds', exist_ok=True)
np.save('../datasets/oof_preds/oof_train_svm.npy', oof_train_svm)
np.save('../datasets/oof_preds/test_preds_svm.npy', test_preds_svm)

In [ ]:
print("Saved SVM OOF and Test predictions successfully!")